In [1]:
# Remove url fields from final_dataset.jsonl
import json
from pathlib import Path

INPUT_JSONL = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final/final_dataset.jsonl")
OUTPUT_JSONL = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final/final_dataset.no_urls.jsonl")

count = 0
with INPUT_JSONL.open("r", encoding="utf-8") as f_in, OUTPUT_JSONL.open("w", encoding="utf-8") as f_out:
    for line in f_in:
        if not line.strip():
            continue
        obj = json.loads(line)
        obj.pop("url", None)
        obj.pop("urls", None)
        f_out.write(json.dumps(obj, ensure_ascii=False) + "\n")
        count += 1

print(f"Wrote {count} records to {OUTPUT_JSONL}")

Wrote 8000 records to /home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final/final_dataset.no_urls.jsonl


In [4]:
# Add metadata column with derived features
import json
import re
from pathlib import Path

INPUT_JSONL = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final/final_dataset.no_urls.jsonl")
OUTPUT_JSONL = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final/final_dataset.with_metadata.jsonl")

FIXED_METADATA_COLUMNS = [
    "num_urls",
    "contains_shortener",
    "contains_ip_url",
    "contains_suspicious_tld",
    "contains_password_language",
    "contains_verification_language",
    "contains_urgency_language",
    "contains_account_language",
    "contains_invoice_language",
    "contains_payment_language",
    "contains_refund_language",
    "contains_banking_language",
    "contains_delivery_language",
    "contains_tracking_language",
    "num_exclamation_marks",
    "num_uppercase_words",
    "contains_call_to_action",
    "has_html_style_text",
    "body_length",
]

URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
IP_URL_RE = re.compile(r"https?://\d{1,3}(?:\.\d{1,3}){3}(?::\d+)?", re.IGNORECASE)
HTML_RE = re.compile(r"<[^>]+>|&nbsp;|style=|font-family|<table|<div|<span", re.IGNORECASE)
UPPER_WORD_RE = re.compile(r"\b[A-Z]{2,}\b")

SHORTENERS = ["bit.ly", "tinyurl.com", "t.co", "goo.gl", "ow.ly", "is.gd", "buff.ly", "adf.ly", "rb.gy"]
SUSPICIOUS_TLDS = [".zip", ".mov", ".click", ".top", ".xyz", ".tk", ".gq", ".cf", ".ml", ".work", ".support", ".ru", ".cn"]

PASSWORD_WORDS = ["password", "passcode", "otp", "pin", "credential"]
VERIFY_WORDS = ["verify", "verification", "authenticate", "authentication", "confirm", "confirmation"]
URGENCY_WORDS = ["urgent", "immediately", "asap", "now", "act now", "final notice", "expires", "deadline"]
ACCOUNT_WORDS = ["account", "login", "sign in", "sign-in", "log in", "password reset"]
INVOICE_WORDS = ["invoice", "inv", "statement"]
PAYMENT_WORDS = ["payment", "paid", "pay now", "billing", "charge"]
REFUND_WORDS = ["refund", "reimbursement", "chargeback"]
BANKING_WORDS = ["bank", "banking", "wire", "transfer", "card", "credit", "debit"]
DELIVERY_WORDS = ["delivery", "parcel", "package", "shipment"]
TRACKING_WORDS = ["tracking", "track", "shipment", "waybill"]
CTA_WORDS = ["click", "verify", "update", "login", "confirm", "act now", "reset", "open", "claim", "view", "download"]

def contains_any(text: str, words: list[str]) -> bool:
    return any(word in text for word in words)

count = 0
with INPUT_JSONL.open("r", encoding="utf-8") as f_in, OUTPUT_JSONL.open("w", encoding="utf-8") as f_out:
    for line in f_in:
        if not line.strip():
            continue
        obj = json.loads(line)
        subject = str(obj.get("subject", ""))
        body = str(obj.get("body", ""))
        combined = f"{subject} {body}"
        combined_lower = combined.lower()

        urls = URL_RE.findall(combined)
        num_urls = len(urls)
        contains_shortener = any(s in combined_lower for s in SHORTENERS)
        contains_ip_url = bool(IP_URL_RE.search(combined))
        contains_suspicious_tld = any(tld in combined_lower for tld in SUSPICIOUS_TLDS)

        metadata = {
            "num_urls": num_urls,
            "contains_shortener": contains_shortener,
            "contains_ip_url": contains_ip_url,
            "contains_suspicious_tld": contains_suspicious_tld,
            "contains_password_language": contains_any(combined_lower, PASSWORD_WORDS),
            "contains_verification_language": contains_any(combined_lower, VERIFY_WORDS),
            "contains_urgency_language": contains_any(combined_lower, URGENCY_WORDS),
            "contains_account_language": contains_any(combined_lower, ACCOUNT_WORDS),
            "contains_invoice_language": contains_any(combined_lower, INVOICE_WORDS),
            "contains_payment_language": contains_any(combined_lower, PAYMENT_WORDS),
            "contains_refund_language": contains_any(combined_lower, REFUND_WORDS),
            "contains_banking_language": contains_any(combined_lower, BANKING_WORDS),
            "contains_delivery_language": contains_any(combined_lower, DELIVERY_WORDS),
            "contains_tracking_language": contains_any(combined_lower, TRACKING_WORDS),
            "num_exclamation_marks": combined.count("!"),
            "num_uppercase_words": len(UPPER_WORD_RE.findall(combined)),
            "contains_call_to_action": contains_any(combined_lower, CTA_WORDS),
            "has_html_style_text": bool(HTML_RE.search(combined)),
            "body_length": len(body),
        }

        # Ensure consistent key order and presence
        obj["metadata"] = {k: metadata.get(k) for k in FIXED_METADATA_COLUMNS}
        f_out.write(json.dumps(obj, ensure_ascii=False) + "\n")
        count += 1

print(f"Wrote {count} records to {OUTPUT_JSONL}")

Wrote 8000 records to /home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final/final_dataset.with_metadata.jsonl


In [10]:
# Convert final dataset to SFT chat format
import json
from pathlib import Path

INPUT_JSONL = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final/final_dataset.with_metadata.jsonl")
OUTPUT_JSONL = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final/final_dataset.sft.jsonl")

SYSTEM_PROMPT = """
You are an expert cybersecurity email classification system.

Your task is to analyze emails and classify them into EXACTLY one of the following categories:

- valid
- spam
- phishing

Category definitions:

valid:
Legitimate personal, business, enterprise, transactional, operational, or opt-in communication from real organizations or individuals.

spam:
Unsolicited junk, low-quality advertising, irrelevant mass messaging, generic scams, or misleading promotional content that does NOT primarily attempt credential theft, impersonation, or account compromise.

phishing:
Malicious or deceptive emails attempting credential theft, impersonation, account compromise, malware delivery, financial fraud, payment fraud, social engineering, or unauthorized access.

If an email contains suspicious wording but lacks clear malicious intent, credential theft, impersonation, or fraud indicators, classify it as spam instead of phishing.

Respond with ONLY the final label.
Do not output additional text.
"""

LABEL_FIELD = "label"
ALLOWED_LABELS = {"valid", "spam", "phishing"}

METADATA_KEYS = [
    "num_urls",
    "contains_shortener",
    "contains_ip_url",
    "contains_suspicious_tld",
    "contains_verification_language",
    "contains_account_language",
    "contains_payment_language",
    "contains_refund_language",
    "contains_delivery_language",
    "contains_tracking_language",
    "num_exclamation_marks",
    "body_length",
 ]

def format_metadata(metadata: dict) -> str:
    lines = []
    for key in METADATA_KEYS:
        lines.append(f"{key}: {metadata.get(key)}")
    return "\n".join(lines)

count = 0
bad_labels = 0

with INPUT_JSONL.open("r", encoding="utf-8") as f_in, OUTPUT_JSONL.open("w", encoding="utf-8") as f_out:
    for line in f_in:
        if not line.strip():
            continue
        obj = json.loads(line)
        label = str(obj.get(LABEL_FIELD, "")).strip().lower()
        if label not in ALLOWED_LABELS:
            bad_labels += 1
            continue

        subject = str(obj.get("subject", "")).strip()
        body = str(obj.get("body", "")).strip()
        metadata = obj.get("metadata", {}) or {}
        filtered_metadata = {key: metadata.get(key) for key in METADATA_KEYS}
        metadata_block = format_metadata(filtered_metadata)
        user_content = (
            "Classify this email.\n\n"
            f"Subject:\n{subject}\n\n"
            f"Metadata:\n{metadata_block}\n\n"
            f"Body:\n{body}"
        )
        record = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": label},
            ]
        }
        f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
        count += 1

print(f"Wrote {count} records to {OUTPUT_JSONL}")
if bad_labels:
    print(f"Skipped {bad_labels} records due to unexpected labels")

Wrote 8000 records to /home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final/final_dataset.sft.jsonl


In [11]:
# Validate JSONL output and locate bad line
import json
from pathlib import Path

CHECK_JSONL = Path("/home/kunal/jhinga/fine-tune/fine-tune-email/dataset_part2/cleaned/extra_clean/final/final_dataset.sft.jsonl")

bad = 0
with CHECK_JSONL.open("r", encoding="utf-8") as f_in:
    for idx, line in enumerate(f_in, start=1):
        try:
            json.loads(line)
        except json.JSONDecodeError as exc:
            print(f"Bad JSON on line {idx}: {exc}")
            print(line[:500])
            bad += 1
            break

if bad == 0:
    print("No JSON errors found.")

No JSON errors found.


In [12]:
# Validate /workspace JSONL and locate bad line
import json
from pathlib import Path

CHECK_JSONL = Path("./final_dataset.sft.jsonl")

bad = 0
with CHECK_JSONL.open("r", encoding="utf-8") as f_in:
    for idx, line in enumerate(f_in, start=1):
        try:
            json.loads(line)
        except json.JSONDecodeError as exc:
            print(f"Bad JSON on line {idx}: {exc}")
            print(line[:500])
            bad += 1
            break

if bad == 0:
    print("No JSON errors found in /workspace file.")

No JSON errors found in /workspace file.
